Subscription Date Overlap Detection
===
Difficulty: Hard

Problem Description:
===================
Given a `subscriptions` table with `start_date` and `end_date` for each user, return True/False
for whether **each user has any two subscriptions whose date ranges overlap**.
Only completed subscriptions (where `end_date` is not null) count.

Sample Input:
```
| user_id | start_date | end_date   |
|---------|------------|------------|
| 1       | 2024-01-01 | 2024-01-31 |
| 1       | 2024-01-15 | 2024-02-15 |  ← overlaps with row 1 ✅
| 2       | 2024-01-01 | 2024-01-31 |
| 2       | 2024-02-01 | 2024-02-28 |  ← sequential, no overlap ❌
```

Sample Output:
```
| user_id | has_overlap |
|---------|-------------|
| 1       | True        |
| 2       | False       |
```

In [2]:
import pandas as pd

subscriptions = pd.DataFrame({
    'id':         range(1, 11),
    'user_id':    [1, 1, 1, 2, 2, 3, 3, 4, 4, 4],
    'start_date': pd.to_datetime([
        '2024-01-01','2024-01-15','2024-03-01',  # User 1: sub1 & sub2 overlap ✅
        '2024-01-01','2024-02-01',               # User 2: sequential ❌
        '2024-01-01','2024-01-01',               # User 3: same start ✅
        '2024-01-01','2024-01-10','2024-01-20',  # User 4: cascading overlaps ✅
    ]),
    'end_date': pd.to_datetime([
        '2024-01-31','2024-02-15','2024-03-31',
        '2024-01-31','2024-02-28',
        '2024-01-15','2024-01-10',
        '2024-01-09','2024-01-19','2024-01-30'
    ])
})
print(subscriptions)

   id  user_id start_date   end_date
0   1        1 2024-01-01 2024-01-31
1   2        1 2024-01-15 2024-02-15
2   3        1 2024-03-01 2024-03-31
3   4        2 2024-01-01 2024-01-31
4   5        2 2024-02-01 2024-02-28
5   6        3 2024-01-01 2024-01-15
6   7        3 2024-01-01 2024-01-10
7   8        4 2024-01-01 2024-01-09
8   9        4 2024-01-10 2024-01-19
9  10        4 2024-01-20 2024-01-30


In [17]:
df = subscriptions
df_sorted = df.sort_values(by=['user_id','start_date'])
df_sorted['next_start_dt'] = df_sorted.groupby('user_id')['start_date'].shift(-1)

df_sorted['has_overlap'] = df_sorted['end_date'] > df_sorted['next_start_dt']

df_sorted
# df_sorted[df_sorted['has_overlap']]

,id,user_id,start_date,end_date,next_start_dt,has_overlap
0,1,1,2024-01-01,2024-01-31,2024-01-15,True
1,2,1,2024-01-15,2024-02-15,2024-03-01,False
2,3,1,2024-03-01,2024-03-31,NaT,False
3,4,2,2024-01-01,2024-01-31,2024-02-01,False
4,5,2,2024-02-01,2024-02-28,NaT,False
5,6,3,2024-01-01,2024-01-15,2024-01-01,True
6,7,3,2024-01-01,2024-01-10,NaT,False
7,8,4,2024-01-01,2024-01-09,2024-01-10,False
8,9,4,2024-01-10,2024-01-19,2024-01-20,False
9,10,4,2024-01-20,2024-01-30,NaT,False


**Concepts to use:**
1. **Two intervals `[s1, e1]` and `[s2, e2]` overlap iff:** `s1 <= e2 AND s2 <= e1`.
2. **Self-join on `user_id`** — merge the table with itself to get all pairs per user, then apply the overlap condition.
3. **Filter `id_x < id_y`** — avoids comparing a row to itself and avoids counting pairs twice.
4. **`groupby().any()`** — True if at least one overlapping pair exists per user.

In [ ]:
# Optimised Solution — Self-join approach
def subscription_overlap(df):
    # Step 1: only completed subscriptions
    df = df.dropna(subset=['end_date']).copy()

    # Step 2: self-join on user_id to get all pairs
    merged = df.merge(df, on='user_id', suffixes=('_a', '_b'))

    # Step 3: exclude same-row pairs and duplicates
    pairs = merged[merged['id_a'] < merged['id_b']]

    # Step 4: overlap condition — [s1 <= e2] AND [s2 <= e1]
    overlapping = pairs[
        (pairs['start_date_a'] <= pairs['end_date_b']) &
        (pairs['start_date_b'] <= pairs['end_date_a'])
    ]

    # Step 5: did ANY pair overlap for each user?
    overlap_users = overlapping.groupby('user_id').size().reset_index(name='overlap_count')
    overlap_users['has_overlap'] = True

    # Step 6: all users, fill False for those with no overlap
    all_users = df[['user_id']].drop_duplicates()
    result = all_users.merge(overlap_users[['user_id','has_overlap']], on='user_id', how='left')
    result['has_overlap'] = result['has_overlap'].fillna(False)
    return result.sort_values('user_id').reset_index(drop=True)

print(subscription_overlap(subscriptions))